In [20]:
import os
import requests
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
%cd /content/drive/MyDrive/stm32_onnx

/content/drive/MyDrive/stm32_onnx


In [22]:
!pip install onnx onnxscript onnxruntime

In [26]:
import torch
import torch.nn as nn
import os
import re
import json
from collections import Counter
from transformers import AutoTokenizer

file_path = 'kobzar.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

text = re.sub(r'[ \t]+', ' ', raw_text)
text = re.sub(r'\n ', '\n', text)
text = re.sub(r'\n{3,}', '\n\n', text)
text = text.strip()

text = text[:len(text) // 24]

VOCAB_SIZE = 1024
vocab_path = "nano_lapa_vocab.json"

lapa_engine = AutoTokenizer.from_pretrained("lapa-llm/tokenizer")

if not os.path.exists(vocab_path):
    print("Analysis for vocab_path...")
    all_tokens = lapa_engine.tokenize(text)
    token_counts = Counter(all_tokens)

    special_tokens = [lapa_engine.pad_token, lapa_engine.unk_token, '\n', ' ']
    base_chars = sorted(list(set(text)))

    reserved = set(special_tokens + base_chars)
    common_lapa = [t for t, c in token_counts.most_common(VOCAB_SIZE) if t not in reserved]

    final_vocab_list = special_tokens + base_chars + common_lapa
    final_vocab_list = final_vocab_list[:VOCAB_SIZE]

    nano_lapa_vocab = {token: i for i, token in enumerate(final_vocab_list)}
    id_to_token = {i: token for token, i in nano_lapa_vocab.items()}

    with open(vocab_path, "w", encoding="utf-8") as f:
        json.dump({"vocab": nano_lapa_vocab, "id_to_token": id_to_token, "vocab_size": len(nano_lapa_vocab)},
                  f, ensure_ascii=False, indent=2)
else:
    with open(vocab_path, "r", encoding="utf-8") as f:
        v_data = json.load(f)
    nano_lapa_vocab = v_data["vocab"]
    id_to_token = v_data["id_to_token"]

vocab_size = len(nano_lapa_vocab)

def nano_lapa_encode(text_to_encode, vocab):
    raw_tokens = lapa_engine.tokenize(text_to_encode)
    ids = []
    for t in raw_tokens:
        if t in vocab:
            ids.append(vocab[t])
        else:
            for char in t:
                ids.append(vocab.get(char, vocab.get("[UNK]", 1)))
    return ids

print("Encoding data...")
encoded_ids = nano_lapa_encode(text, nano_lapa_vocab)
data = torch.tensor(encoded_ids, dtype=torch.long)

print(f"Success. Vocab: {vocab_size} tokens. Tensor data ready.")
print(f"Tokens count: {len(data)}")

Encoding data...
Success. Vocab: 1022 tokens. Tensor data ready.
Tokens count: 13098


In [27]:
# ==========================================
# 2. АРХІТЕКТУРА NANO-TRANSFORMER (STATIC-READY)
# ==========================================

# Гіперпараметри (Вже під Nano-Lapa)
SEQ_LEN = 32
BATCH_SIZE = 128
D_MODEL = 32
N_HEADS = 4
N_LAYERS = 2
FF_DIM = 64
DROPOUT = 0.1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class StaticAttention(nn.Module):
    """Кастомний статичний Attention для ідеального ONNX експорту на STM32"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(d_model, d_model * 3)
        self.proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = torch.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = (attn @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class NanoBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = StaticAttention(d_model, n_heads, dropout=dropout)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.ReLU6(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model),
            nn.Dropout(dropout)
        )
        self.bn1 = nn.BatchNorm1d(d_model)
        self.bn2 = nn.BatchNorm1d(d_model)

    def forward(self, x):
        B, T, C = x.shape
        # 1. Attention + Residual
        x = x + self.attention(x)
        # BatchNorm трюк
        x = self.bn1(x.reshape(-1, C)).reshape(B, T, C)

        # 2. Feed Forward + Residual
        x = x + self.feed_forward(x)
        # BatchNorm трюк
        x = self.bn2(x.reshape(-1, C)).reshape(B, T, C)
        return x

class NanoTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, ff_dim, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, max_seq_len, d_model))

        self.blocks = nn.ModuleList([
            NanoBlock(d_model, n_heads, ff_dim, DROPOUT) for _ in range(n_layers)
        ])

        self.fc_out = nn.Linear(d_model, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

    def forward(self, x):
        # x: (B, T)
        x = x.long()
        x = self.embedding(x) + self.pos_emb[:, :x.size(1), :]

        for block in self.blocks:
            x = block(x)

        # Статична індексація останнього токена для X-CUBE-AI
        # Це змушує ONNX завжди дивитися на останній елемент у вікні 64
        out = x[:, self.max_seq_len - 1, :]
        logits = self.fc_out(out)
        return logits

vocab_size = int(data.max().item() + 1)
print(f" Остаточний Vocab Size: {vocab_size}") # МАЄ ВИВЕСТИ 1024!


model = NanoTransformer(
    vocab_size=vocab_size,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    ff_dim=FF_DIM,
    max_seq_len=SEQ_LEN  #
).to(device)

print(" Модель ініціалізована з Vocab=1024. CUDA помилок не буде!")

 Остаточний Vocab Size: 1024
 Модель ініціалізована з Vocab=1024. CUDA помилок не буде!


In [ ]:
import os

# Створюємо папку для моделей, якщо її немає
SAVE_DIR = "saved_models"
os.makedirs(SAVE_DIR, exist_ok=True)

def get_batch():
    # Використовуємо глобальний SEQ_LEN (64)
    # Нам потрібно SEQ_LEN токенів для X і один наступний для Y
    max_idx = len(data) - SEQ_LEN - 1
    if max_idx <= 0:
        raise ValueError(f"Текст занадто короткий для SEQ_LEN={SEQ_LEN}!")

    # Вибираємо випадкові індекси для батчу
    ix = torch.randint(0, max_idx, (BATCH_SIZE,))

    # Формуємо вхідні послідовності (B, SEQ_LEN)
    x = torch.stack([data[i:i+SEQ_LEN] for i in ix])
    # Цільовий токен - це ЗАВЖДИ наступний токен після вікна SEQ_LEN
    y = torch.stack([data[i+SEQ_LEN] for i in ix])

    return x.to(device), y.to(device)

# Налаштування навчання
# label_smoothing допомагає моделі не ставати занадто самовпевненою (корисно для малих моделей)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.008, weight_decay=0.05) # Трохи зменшив LR для стабільності 1024 токенів

iterations = 30000
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=iterations)

# --- Параметри Early Stopping ---
best_loss = float('inf')
patience = 12
patience_counter = 0
min_delta = 0.001

print(f"Починаємо тренування на {device}...")
print(f"Конфігурація: Vocab={vocab_size}, D_Model={D_MODEL}, Seq_Len={SEQ_LEN}")
model.train()

for step in range(iterations):
    Xb, Yb = get_batch()

    logits = model(Xb) # На виході (B, vocab_size)
    loss = criterion(logits, Yb)

    optimizer.zero_grad()
    loss.backward()

    # Градієнтний кліпінг - обов'язково для Трансформерів!
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()
    scheduler.step()

    # Перевірка прогресу кожні 500 кроків
    if step % 500 == 0:
        current_loss = loss.item()
        lr = scheduler.get_last_lr()[0]
        print(f"Крок {step:05d} | Loss: {current_loss:.4f} | LR: {lr:.6f}")

        # Логіка збереження найкращої моделі
        if current_loss < (best_loss - min_delta):
            best_loss = current_loss
            patience_counter = 0

            checkpoint_path = os.path.join(SAVE_DIR, "best_kobzar_model.pth")
            torch.save(model.state_dict(), checkpoint_path)
            print(f"Знайдено кращу модель! Збережено: {checkpoint_path}")
        else:
            patience_counter += 1
            print(f"Покращення відсутнє {patience_counter}/{patience}")

        # Early Stopping
        if patience_counter >= patience:
            print(f"Early Stopping активовано на кроці {step}. Модель досягла свого піку.")
            break

print(f"🏁 Тренування завершено! Найкращий Loss: {best_loss:.4f}")

Починаємо тренування на cuda...
Конфігурація: Vocab=1024, D_Model=32, Seq_Len=32
Крок 00000 | Loss: 2.8046 | LR: 0.008000
Знайдено кращу модель! Збережено: saved_models/best_kobzar_model.pth
Крок 00500 | Loss: 3.1328 | LR: 0.007994
Покращення відсутнє 1/12
Крок 01000 | Loss: 3.2441 | LR: 0.007978
Покращення відсутнє 2/12
Крок 01500 | Loss: 3.2859 | LR: 0.007951
Покращення відсутнє 3/12
Крок 02000 | Loss: 2.9511 | LR: 0.007913
Покращення відсутнє 4/12
Крок 02500 | Loss: 2.9898 | LR: 0.007864
Покращення відсутнє 5/12
Крок 03000 | Loss: 3.0035 | LR: 0.007804
Покращення відсутнє 6/12
Крок 03500 | Loss: 3.0769 | LR: 0.007734
Покращення відсутнє 7/12
Крок 04000 | Loss: 2.8866 | LR: 0.007654
Покращення відсутнє 8/12
Крок 04500 | Loss: 2.6723 | LR: 0.007564
Знайдено кращу модель! Збережено: saved_models/best_kobzar_model.pth
Крок 05000 | Loss: 3.0972 | LR: 0.007464
Покращення відсутнє 1/12
Крок 05500 | Loss: 2.8507 | LR: 0.007354
Покращення відсутнє 2/12
Крок 06000 | Loss: 2.8772 | LR: 0.00723

In [ ]:
import torch
import onnx
import os

# ==========================================
# 4. ЕКСПОРТ ТА ОЧИСТКА ONNX ДЛЯ STM32
# ==========================================

# Шлях до нашої найкращої моделі
checkpoint_path = os.path.join(SAVE_DIR, "best_kobzar_model.pth")


model.load_state_dict(torch.load(checkpoint_path, map_location=device))


model.eval()



# --- 2. ЕКСПОРТ В ONNX ---
model_path = "kobzar_static.onnx"

dummy_input = torch.zeros((1, SEQ_LEN), dtype=torch.int32).to(device)

print(f" Експортуємо PyTorch модель у {model_path}...")
torch.onnx.export(
    model,                      # Наша навчена модель
    dummy_input,                # Фіктивний вхід для трасування
    model_path,                 # Куди зберігати
    export_params=True,         # Зберігати ваги всередині файлу
    opset_version=11,           # Версія 11 ідеально підходить для X-CUBE-AI
    do_constant_folding=True,   # Оптимізація констант
    input_names=['input'],      # Ім'я входу для С-коду
    output_names=['output']     # Ім'я виходу для С-коду
)

# --- 3. ОЧИСТКА ONNX ВІД ALLOWZERO (Твій код) ---
onnx_model = onnx.load(model_path)

fixed_nodes = 0
for node in onnx_model.graph.node:
    if node.op_type == "Reshape":
        # Видаляємо атрибут allowzero, залишаючи всі інші
        clean_attributes = [attr for attr in node.attribute if attr.name != "allowzero"]
        if len(clean_attributes) < len(node.attribute):
            del node.attribute[:]
            node.attribute.extend(clean_attributes)
            fixed_nodes += 1

clean_model_path = "kobzar_clean.onnx"
onnx.save(onnx_model, clean_model_path)


In [ ]:
import onnxruntime as ort
import numpy as np
import json

def c_like_tokenize(text, vocab):
    """
    Алгоритм токенізації для мікроконтролера (Жадібний пошук).
    Легко переписується на C за допомогою strncmp.
    """
    # Lapa використовує спецсимволи для пробілів.
    # Підміняємо звичайний пробіл на той маркер, що є в словнику.
    # Найчастіше це "▁" (U+25C1) або " " (U+2581).
    if "▁" in vocab or any(k.startswith("▁") for k in vocab.keys()):
        text = text.replace(" ", "▁")
    elif " " in vocab or any(k.startswith(" ") for k in vocab.keys()):
        text = text.replace(" ", " ")

    ids = []
    i = 0
    text_len = len(text)

    while i < text_len:
        match_found = False
        for j in range(text_len, i, -1):
            sub_str = text[i:j]
            if sub_str in vocab:
                ids.append(vocab[sub_str])
                i = j
                match_found = True
                break

        if not match_found:
            ids.append(vocab.get("[UNK]", 1))
            i += 1

    return ids

def debug_stm32_logic(onnx_file, prompt_text, temperature=0.6):
    with open("nano_lapa_vocab.json", "r", encoding="utf-8") as f:
        vocab_data = json.load(f)

    vocab = vocab_data["vocab"]
    id_to_token = vocab_data["id_to_token"]

    # 2. КОДУВАННЯ ПРОМПТУ (ЧИСТА ЛОГІКА)
    prompt_indices = c_like_tokenize(prompt_text, vocab)
    print(f"Токени промпту: {prompt_indices}")

    # 3. НАЛАШТУВАННЯ ONNX СЕСІЇ
    session = ort.InferenceSession(onnx_file)
    input_name = session.get_inputs()[0].name
    expected_seq_len = session.get_inputs()[0].shape[1]

    print(f"--- МОДЕЛЬ: {onnx_file} | SEQ_LEN: {expected_seq_len} | TEMP: {temperature} ---")

    pad_id = vocab.get("[PAD]", 0)

    # Створюємо вікно контексту
    my_context_window = [pad_id] * expected_seq_len

    # Вставляємо промпт
    p_len = len(prompt_indices)
    for i in range(min(p_len, expected_seq_len)):
        my_context_window[expected_seq_len - min(p_len, expected_seq_len) + i] = prompt_indices[-min(p_len, expected_seq_len) + i]

    print(f"--- ГЕНЕРАЦІЯ ПОЧАЛАСЯ ---")

    generated_ids = []

    for step in range(100):
        # Тип int32, бо експортували саме так
        input_tensor = np.array([my_context_window], dtype=np.int32)
        outputs = session.run(None, {input_name: input_tensor})
        logits = outputs[0]

        last_logits = logits[0]

        # --- TEMPERATURE SAMPLING ---
        last_logits = last_logits / max(temperature, 1e-6)
        exp_logits = np.exp(last_logits - np.max(last_logits))
        probs = exp_logits / np.sum(exp_logits)

        next_id = np.random.choice(len(probs), p=probs)
        # ----------------------------

        generated_ids.append(next_id)

        # Зсуваємо вікно
        my_context_window = my_context_window[1:] + [next_id]

    # 4. ДЕКОДУВАННЯ (Очистка тексту)
    raw_output = "".join([id_to_token.get(str(i), "") for i in generated_ids])
    final_text = raw_output.replace(" ", " ").replace("▁", " ").replace("[PAD]", "").replace("[UNK]", "")

    print("\n" + "="*40)
    print("ВХІДНИЙ ПРОМПТ:")
    print(prompt_text)
    print("-" * 20)
    print("ПРОДОВЖЕННЯ МОДЕЛЛЮ:")
    print(final_text)
    print("="*40)

# Запуск
debug_stm32_logic('kobzar_clean.onnx', "Думи мої, думи мої, ", temperature=0.3)

In [ ]:
import numpy as np
import onnxruntime as ort
import json
import os

def generate_cubeai_validation_data(onnx_path, vocab_path, num_samples=50):

    # 1. Читаємо розмір словника (щоб не згенерувати індекс, якого не існує)
    with open(vocab_path, "r", encoding="utf-8") as f:
        vocab_data = json.load(f)
    vocab_size = vocab_data["vocab_size"]

    # 2. Генеруємо вхідні дані
    # X-CUBE-AI очікує масив розміром (Кількість_прикладів, 64)
    # Зверни увагу: тип int32, як і очікує наша оптимізована модель
    SEQ_LEN = 32
    val_input = np.random.randint(0, vocab_size, size=(num_samples, SEQ_LEN), dtype=np.int32)

    # 3. Запускаємо ONNX Runtime, щоб отримати "еталонні" виходи
    sess = ort.InferenceSession(onnx_path)
    input_name = sess.get_inputs()[0].name

    val_output = []

    print("Прорахунок еталонних виходів через ONNX...")
    for i in range(num_samples):
        # Беремо один приклад, робимо йому форму (1, 64)
        single_in = val_input[i:i+1]
        out = sess.run(None, {input_name: single_in})[0]
        val_output.append(out[0]) # Додаємо вихід (розміром 1024)

    # Перетворюємо список у numpy-масив
    val_output = np.array(val_output, dtype=np.float32)

    # 4. Зберігаємо файли
    in_file = "cube_val_input.npy"
    out_file = "cube_val_output.npy"

    np.save(in_file, val_input)
    np.save(out_file, val_output)

    print(f"Validation input:  {in_file}  {val_input.shape}")
    print(f"Validation output: {out_file} {val_output.shape}")

# Запуск
generate_cubeai_validation_data("kobzar_clean.onnx", "nano_lapa_vocab.json", num_samples=50)

In [ ]:
import json
from transformers import AutoTokenizer

def generate_c_header():
    # Завантажуємо дані
    lapa_engine = AutoTokenizer.from_pretrained("lapa-llm/tokenizer")
    with open("nano_lapa_vocab.json", "r", encoding="utf-8") as f:
        v_data = json.load(f)

    vocab = v_data["vocab"]
    id_to_token = v_data["id_to_token"]

    # 1. Готуємо промпт
    prompt_text = "Думи мої, думи мої, "
    raw_tokens = lapa_engine.tokenize(prompt_text)
    prompt_indices = [vocab.get(t, vocab.get("[UNK]", 1)) for t in raw_tokens]

    # 2. Генеруємо C-код
    c_code = "#ifndef VOCAB_H\n#define VOCAB_H\n\n#include <stdint.h>\n\n"
    c_code += f"#define PROMPT_LEN {len(prompt_indices)}\n"
    c_code += f"static const int32_t prompt_array[PROMPT_LEN] = {{{', '.join(map(str, prompt_indices))}}};\n\n"

    c_code += "static const char* ix_to_char_map[1024] = {\n"

    for i in range(1024):
        token = id_to_token.get(str(i), "")
        # Очистка від спецсимволів (замінюємо маркер Lapa на пробіл)
        token = token.replace("▁", " ").replace("Ġ", " ")
        # Прибираємо технічні теги, щоб вони не друкувалися в консоль
        if token in ["[PAD]", "[UNK]", "<s>", "</s>"]: token = ""

        # Екранування для C-коду
        token = token.replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n")
        c_code += f'    "{token}", // {i}\n'

    c_code += "};\n\n#endif // VOCAB_H\n"

    # Зберігаємо
    with open("vocab.h", "w", encoding="utf-8") as f:
        f.write(c_code)


generate_c_header()